In [2]:
%pip install scikit-learn rank-bm25 sentence-transformers numpy

  Using cached rank_bm25-0.2.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached sentence_transformers-6.0.1-py3-none-any.whl.metadata (20 kB)
Using cached rank_bm25-0.2.2-py3-none-any.whl (8.6 kB)
Using cached sentence_transformers-6.0.1-py3-none-any.whl (739 kB)

   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -----------


[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: C:\Users\Marwa Sayed\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Load Libraries

In [3]:
import json
import re 
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer


C:\Users\Marwa Sayed\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## load DataSet

In [4]:
def load_corpus(path):
    papers = []

    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            paper = json.loads(line)
            papers.append(paper)

    return papers

In [5]:
corpus = load_corpus("corpus.jsonl")

print(len(corpus))
corpus[0]

5183


{'doc_id': 4983,
 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.',
 'abstract': ['Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities.',
  'A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7).',
  'To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term.',
  'In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms.',
  'In the posterior limb 

In [6]:
print(corpus[0]["doc_id"])
print(corpus[0]["title"])
print(corpus[0]["abstract"])

4983
Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
['Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities.', 'A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7).', 'To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term.', 'In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms.', 'In the posterior limb of the internal capsule, the mean apparent dif

## Data Inspection

In [7]:
print("Corpus abstract type : ", type(corpus[0]["abstract"]))

#sentence number 
print("sentence number : " , len(corpus[0]["abstract"]))

print("First sentence : ", corpus[0]["abstract"][0])

Corpus abstract type :  <class 'list'>
sentence number :  14
First sentence :  Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities.


In [8]:
def prepare_document(paper):
    title = paper["title"]
    abstract = " ".join(paper["abstract"])
    document_text = title + " " + abstract

    return document_text

document = prepare_document(corpus[0])

document

'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging. Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, the mean apparent diffusion coefficient

In [9]:
#make the whole corpus 
documents = []

for paper in corpus:
    document = prepare_document(paper)
    documents.append(document)

print(len(documents))

5183


In [10]:
documents[0]

'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging. Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, the mean apparent diffusion coefficient

##  TF-IDF Retrieval

In [16]:
class TFIDFRetriever:

    def __init__(self, corpus):
        self.corpus = corpus

        self.documents = documents

        self.vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2)
        )

        self.document_vectors = self.vectorizer.fit_transform(
            self.documents
        )

    def retrieve(self, claim, top_k=5):

        claim_vector = self.vectorizer.transform([claim])

        scores = cosine_similarity(
            claim_vector,
            self.document_vectors
        )[0]

        top_indices = np.argsort(scores)[::-1][:top_k]

        results = []

        for index in top_indices:

            paper = self.corpus[index]

            results.append({
                "doc_id": paper["doc_id"],
                "title": paper["title"],
                "score": float(scores[index]),
                "abstract": paper["abstract"]
            })

        return results

In [ ]:
#test Tf-idf
corpus = load_corpus("corpus.jsonl")

tfidf_retriever = TFIDFRetriever(corpus)

claim = "Prematurity affects cerebral white matter development."

results = tfidf_retriever.retrieve(
        claim,
        top_k=5
    )

print("\n===== TF-IDF RESULTS =====")

for result in results:

    print("\nDoc ID:", result["doc_id"])
    print("Title:", result["title"])
    print("Score:", result["score"])


===== TF-IDF RESULTS =====

Doc ID: 4983
Title: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
Score: 0.408817003859458

Doc ID: 1412089
Title: Diminished performance on neuropsychological testing in late life depression is correlated with microstructural white matter abnormalities.
Score: 0.2506619103513072

Doc ID: 22107641
Title: Late-life depression and microstructural abnormalities in dorsolateral prefrontal cortex white matter.
Score: 0.187248318336902

Doc ID: 17930286
Title: Headache, migraine, and structural brain lesions and function: population based Epidemiology of Vascular Ageing-MRI study
Score: 0.14636511997231055

Doc ID: 1472815
Title: Alterations of white matter integrity in adults with major depressive disorder: a magnetic resonance imaging study.
Score: 0.14410076468240382


## BM25Retriever

In [18]:
def tokenize(text):
    text = text.lower()
    tokens = re.findall(
        r"\b[a-zA-Z]+\b",
        text
    )

    return tokens

In [19]:
class BM25Retriever:

    def __init__(self, corpus):

        self.corpus = corpus

        self.documents = documents

        self.tokenized_documents = [
            tokenize(document)
            for document in self.documents
        ]

        self.bm25 = BM25Okapi(
            self.tokenized_documents
        )

    def retrieve(self, claim, top_k=5):

        query_tokens = tokenize(claim)

        scores = self.bm25.get_scores(
            query_tokens
        )

        top_indices = np.argsort(scores)[::-1][:top_k]

        results = []

        for index in top_indices:

            paper = self.corpus[index]

            results.append({
                "doc_id": paper["doc_id"],
                "title": paper["title"],
                "score": float(scores[index]),
                "abstract": paper["abstract"]
            })

        return results

In [ ]:
#test BM25
corpus = load_corpus("corpus.jsonl")

tfidf_retriever = BM25Retriever(corpus)

claim = "Prematurity affects cerebral white matter development."

results = tfidf_retriever.retrieve(
        claim,
        top_k=5
    )

print("\n===== TF-IDF RESULTS =====")

for result in results:

    print("\nDoc ID:", result["doc_id"])
    print("Title:", result["title"])
    print("Score:", result["score"])


===== TF-IDF RESULTS =====

Doc ID: 4983
Title: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
Score: 36.59155958791644

Doc ID: 8227227
Title: Locations of cerebral infarctions in tuberculous meningitis
Score: 20.258535179588897

Doc ID: 1412089
Title: Diminished performance on neuropsychological testing in late life depression is correlated with microstructural white matter abnormalities.
Score: 18.356174616468792

Doc ID: 22107641
Title: Late-life depression and microstructural abnormalities in dorsolateral prefrontal cortex white matter.
Score: 17.695535039671007

Doc ID: 18104691
Title: Neurological outcomes of animal models of uterine artery ligation and relevance to human intrauterine growth restriction: a systematic review
Score: 17.67830572816744


## Semantic retrieval

In [22]:
class SemanticRetriever:

    def __init__(self, corpus):

        self.corpus = corpus

        self.documents = documents

        print("Loading semantic model...")

        self.model = SentenceTransformer(
            "allenai-specter"
        )

        print("Creating document embeddings...")

        self.document_embeddings = self.model.encode(
            self.documents,
            convert_to_numpy=True,
            show_progress_bar=True
        )

    def retrieve(self, claim, top_k=5):

        claim_embedding = self.model.encode(
            [claim],
            convert_to_numpy=True
        )

        scores = cosine_similarity(
            claim_embedding,
            self.document_embeddings
        )[0]

        top_indices = np.argsort(scores)[::-1][:top_k]

        results = []

        for index in top_indices:

            paper = self.corpus[index]

            results.append({
                "doc_id": paper["doc_id"],
                "title": paper["title"],
                "score": float(scores[index]),
                "abstract": paper["abstract"]
            })

        return results

In [26]:
semantic_retriever = SemanticRetriever(corpus)

Loading semantic model...


C:\Users\Marwa Sayed\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Marwa Sayed\.cache\huggingface\hub\models--sentence-transformers--allenai-specter. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message

Creating document embeddings...


Batches: 100%|██████████| 162/162 [34:28<00:00, 12.77s/it]


In [27]:
def save_embeddings(embeddings, path):

    np.save(path, embeddings)


def load_embeddings(path):

    return np.load(path)

In [ ]:
#save the embeddings
save_embeddings(semantic_retriever.document_embeddings,"document_embeddings.npy")

In [29]:
semantic_retriever.document_embeddings = \
    load_embeddings("document_embeddings.npy")

## Evaluate the Retrieval algorithms 

In [33]:
claim = "Prematurity affects cerebral white matter development."

# TF-IDF
print("\n\n===== TF-IDF =====")

tfidf_retriever = TFIDFRetriever(corpus)

tfidf_results = tfidf_retriever.retrieve(
    claim,
    top_k=5
)

for result in tfidf_results:

    print("\nDoc ID:", result["doc_id"])
    print("Title:", result["title"])
    print("Score:", result["score"])





===== TF-IDF =====

Doc ID: 4983
Title: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
Score: 0.408817003859458

Doc ID: 1412089
Title: Diminished performance on neuropsychological testing in late life depression is correlated with microstructural white matter abnormalities.
Score: 0.2506619103513072

Doc ID: 22107641
Title: Late-life depression and microstructural abnormalities in dorsolateral prefrontal cortex white matter.
Score: 0.187248318336902

Doc ID: 17930286
Title: Headache, migraine, and structural brain lesions and function: population based Epidemiology of Vascular Ageing-MRI study
Score: 0.14636511997231055

Doc ID: 1472815
Title: Alterations of white matter integrity in adults with major depressive disorder: a magnetic resonance imaging study.
Score: 0.14410076468240382


In [34]:
# BM25

print("\n\n===== BM25 =====")

bm25_retriever = BM25Retriever(corpus)

bm25_results = bm25_retriever.retrieve(
    claim,
    top_k=5
)

for result in bm25_results:

    print("\nDoc ID:", result["doc_id"])
    print("Title:", result["title"])
    print("Score:", result["score"])




===== BM25 =====

Doc ID: 4983
Title: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
Score: 36.59155958791644

Doc ID: 8227227
Title: Locations of cerebral infarctions in tuberculous meningitis
Score: 20.258535179588897

Doc ID: 1412089
Title: Diminished performance on neuropsychological testing in late life depression is correlated with microstructural white matter abnormalities.
Score: 18.356174616468792

Doc ID: 22107641
Title: Late-life depression and microstructural abnormalities in dorsolateral prefrontal cortex white matter.
Score: 17.695535039671007

Doc ID: 18104691
Title: Neurological outcomes of animal models of uterine artery ligation and relevance to human intrauterine growth restriction: a systematic review
Score: 17.67830572816744


In [35]:
# SEMANTIC
print("\n\n===== SEMANTIC RETRIEVAL =====")

#semantic_retriever = SemanticRetriever(corpus)

semantic_results = semantic_retriever.retrieve(
    claim,
    top_k=5
)

for result in semantic_results:

    print("\nDoc ID:", result["doc_id"])
    print("Title:", result["title"])
    print("Score:", result["score"])



===== SEMANTIC RETRIEVAL =====

Doc ID: 13791044
Title: Cerebral palsy among term and postterm births.
Score: 0.8298076391220093

Doc ID: 4983
Title: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
Score: 0.819908618927002

Doc ID: 33257464
Title: Changes in the prevalence of cerebral palsy for children born very prematurely within a population-based program over 30 years.
Score: 0.8026962280273438

Doc ID: 2613775
Title: Sudden infant death syndrome.
Score: 0.7960163354873657

Doc ID: 28633594
Title: International standards for fetal growth based on serial ultrasound measurements: the Fetal Growth Longitudinal Study of the INTERGROWTH-21st Project.
Score: 0.7836053371429443


## Evalution metrics 

In [ ]:
def get_gold_documents(claim):

    evidence = claim.get("evidence", {})

    gold_docs = set()

    for doc_id in evidence.keys():

        gold_docs.add(int(doc_id))

    return gold_docs




def recall_at_k(retrieved_results, gold_docs, k):

    retrieved_docs = {
        result["doc_id"]
        for result in retrieved_results[:k]
    }

    if len(gold_docs.intersection(retrieved_docs)) > 0:

        return 1

    return 0


def reciprocal_rank(retrieved_results, gold_docs):

    for rank, result in enumerate(
        retrieved_results,
        start=1
    ):

        if result["doc_id"] in gold_docs:

            return 1 / rank

    return 0

In [39]:
def evaluate_retriever(
    retriever,
    claims,
    max_k=10
):

    recall_1 = []
    recall_5 = []
    recall_10 = []
    reciprocal_ranks = []

    for claim in claims:

        gold_docs = get_gold_documents(claim)

        if not gold_docs:
            continue

        results = retriever.retrieve(
            claim["claim"],
            top_k=max_k
        )

        recall_1.append(
            recall_at_k(
                results,
                gold_docs,
                1
            )
        )

        recall_5.append(
            recall_at_k(
                results,
                gold_docs,
                5
            )
        )

        recall_10.append(
            recall_at_k(
                results,
                gold_docs,
                10
            )
        )

        reciprocal_ranks.append(
            reciprocal_rank(
                results,
                gold_docs
            )
        )

    return {
        "Recall@1": np.mean(recall_1),
        "Recall@5": np.mean(recall_5),
        "Recall@10": np.mean(recall_10),
        "MRR": np.mean(reciprocal_ranks)
    }

In [ ]:
def load_claims(path):

    with open(path, "r", encoding="utf-8") as file:

        content = file.read().strip()

    # JSON
    if content.startswith("["):

        return json.loads(content)

    # JSONL
    claims = []

    for line in content.splitlines():

        if line.strip():

            claims.append(
                json.loads(line)
            )

    return claims


claims = load_claims(
    "claims_dev.jsonl"
)

In [45]:
print("\n===== EVALUATION =====")

tfidf_metrics = evaluate_retriever(
    tfidf_retriever,
    claims
)

bm25_metrics = evaluate_retriever(
    bm25_retriever,
    claims
)

semantic_metrics = evaluate_retriever(
    semantic_retriever,
    claims
)

print("\nTF-IDF:")
print(tfidf_metrics)

print("\nBM25:")
print(bm25_metrics)

print("\nSemantic:")
print(semantic_metrics)


===== EVALUATION =====

TF-IDF:
{'Recall@1': np.float64(0.601063829787234), 'Recall@5': np.float64(0.8351063829787234), 'Recall@10': np.float64(0.8829787234042553), 'MRR': np.float64(0.6919537318473488)}

BM25:
{'Recall@1': np.float64(0.6436170212765957), 'Recall@5': np.float64(0.8404255319148937), 'Recall@10': np.float64(0.898936170212766), 'MRR': np.float64(0.7274864910503207)}

Semantic:
{'Recall@1': np.float64(0.40425531914893614), 'Recall@5': np.float64(0.6861702127659575), 'Recall@10': np.float64(0.7553191489361702), 'MRR': np.float64(0.523032759202972)}
